# JEPA-to-CLIP Embedding Pre-computation

Downloads MSR-VTT videos, runs V-JEPA and CLIP inference, saves paired embeddings to HDF5.

**Run on a GPU runtime** (Colab T4/A100 or similar). Downloads ~6GB data + ~3GB models.

In [1]:
!pip install -q torch torchvision transformers h5py av tqdm Pillow huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 15.8 MB/s eta 0:00:00


In [2]:
import torch
print(f"torch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    DEVICE = "cuda"
else:
    print("No GPU — will be slow but functional")
    DEVICE = "cpu"

torch 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 1. Download MSR-VTT

In [3]:
if not SKIP_PRECOMPUTE:
    import os
    from huggingface_hub import hf_hub_download

    DATA_DIR = "data"
    os.makedirs(f"{DATA_DIR}/videos", exist_ok=True)

    # Download the video archive (~4.3GB)
    zip_path = hf_hub_download(
        repo_id="AlexZigma/msr-vtt",
        filename="data/MSR-VTT.ZIP",
        repo_type="dataset",
        local_dir=DATA_DIR,
    )
    print(f"Downloaded to: {zip_path}")
else:
    print("Skipped (embeddings already exist)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


data/MSR-VTT.ZIP:   0%|          | 0.00/4.58G [00:00<?, ?B/s]

Downloaded to: data/data/MSR-VTT.ZIP


In [4]:
if not SKIP_PRECOMPUTE:
    import zipfile

    # Extract videos
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)

    # Symlink videos to expected location
    video_src = os.path.join(DATA_DIR, "MSR-VTT", "train_val_videos", "TrainValVideo")
    video_dst = os.path.join(DATA_DIR, "videos")

    # Move mp4 files into videos/ dir
    import shutil
    for f in os.listdir(video_src):
        if f.endswith(".mp4"):
            src = os.path.join(video_src, f)
            dst = os.path.join(video_dst, f)
            if not os.path.exists(dst):
                shutil.move(src, dst)

    # Copy annotations
    ann_src = os.path.join(DATA_DIR, "MSR-VTT", "train_val_annotation", "train_val_videodatainfo.json")
    ann_dst = os.path.join(DATA_DIR, "annotations.json")
    shutil.copy2(ann_src, ann_dst)

    num_videos = len([f for f in os.listdir(video_dst) if f.endswith(".mp4")])
    print(f"Extracted {num_videos} videos + annotations")
else:
    print("Skipped (embeddings already exist)")

Extracted 7010 videos + annotations


## 2. Configuration

In [5]:
CONFIG = {
    "vjepa_model": "facebook/vjepa2-vitl-fpc64-256",
    "clip_model": "openai/clip-vit-large-patch14",
    "subset_size": 500,        # number of videos to process (set to None for all)
    "max_frames": 64,          # frames per video for V-JEPA
    "clip_sample_frames": 8,   # frames to sample for CLIP image encoder
    "max_captions": 5,         # captions per video to encode
    "output_path": "msrvtt_embeddings.h5",
    "device": DEVICE,
}
print(f"Will process {CONFIG['subset_size'] or 'ALL'} videos on {CONFIG['device']}")

Will process 500 videos on cuda


In [ ]:
import os

# Skip precompute steps if embeddings already exist
EMBEDDINGS_FILE = CONFIG.get("output_path", "msrvtt_embeddings.h5")
SKIP_PRECOMPUTE = os.path.exists(EMBEDDINGS_FILE)
if SKIP_PRECOMPUTE:
    print(f"✓ Found existing embeddings at {EMBEDDINGS_FILE} — skipping cells 3-8 (download, models, compute)")
    print("  Jump straight to Part 2 (training) cells below.")
else:
    print(f"No embeddings found at {EMBEDDINGS_FILE} — will run full precompute pipeline")

## 3. Load Annotations & Parse Captions

In [6]:
if not SKIP_PRECOMPUTE:
    import json

    with open(os.path.join(DATA_DIR, "annotations.json")) as f:
        ann_data = json.load(f)

    videos = ann_data["videos"]
    captions = {}
    for sent in ann_data["sentences"]:
        vid_id = sent["video_id"]
        captions.setdefault(vid_id, []).append(sent["caption"])

    # Filter to videos that exist locally
    available_ids = set()
    for f_name in os.listdir(os.path.join(DATA_DIR, "videos")):
        if f_name.endswith(".mp4"):
            available_ids.add(f_name.replace(".mp4", ""))

    video_ids = [v["video_id"] for v in videos if v["video_id"] in available_ids]
    if CONFIG["subset_size"]:
        video_ids = video_ids[:CONFIG["subset_size"]]

    print(f"Available: {len(available_ids)} videos")
    print(f"Processing: {len(video_ids)} videos")
    print(f"Sample captions for {video_ids[0]}: {captions[video_ids[0]][:2]}")
else:
    print("Skipped (embeddings already exist)")

Available: 7010 videos
Processing: 500 videos
Sample captions for video0: ['a car is shown', 'a group is dancing']


## 4. Load Models

In [7]:
if not SKIP_PRECOMPUTE:
    from transformers import AutoModel, AutoVideoProcessor, CLIPModel, CLIPProcessor

    device = torch.device(CONFIG["device"])

    # V-JEPA (1024-dim embeddings)
    print("Loading V-JEPA...")
    vjepa_processor = AutoVideoProcessor.from_pretrained(CONFIG["vjepa_model"])
    vjepa_model = AutoModel.from_pretrained(CONFIG["vjepa_model"], torch_dtype=torch.float16)
    vjepa_model = vjepa_model.to(device).eval()
    for p in vjepa_model.parameters():
        p.requires_grad = False
    vjepa_dim = vjepa_model.config.hidden_size
    print(f"V-JEPA loaded: {sum(p.numel() for p in vjepa_model.parameters())/1e6:.0f}M params, {vjepa_dim}-dim")

    # CLIP (768-dim embeddings)
    print("Loading CLIP...")
    clip_processor = CLIPProcessor.from_pretrained(CONFIG["clip_model"])
    clip_model = CLIPModel.from_pretrained(CONFIG["clip_model"], torch_dtype=torch.float16)
    clip_model = clip_model.to(device).eval()
    for p in clip_model.parameters():
        p.requires_grad = False
    clip_dim = clip_model.config.projection_dim
    print(f"CLIP loaded: {sum(p.numel() for p in clip_model.parameters())/1e6:.0f}M params, {clip_dim}-dim")

    if torch.cuda.is_available():
        print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f}GB")
else:
    print("Skipped (embeddings already exist)")

Loading V-JEPA...


video_preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/785 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.30G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP loaded: 428M params, 768-dim
GPU memory used: 1.5GB


## 5. Define Embedding Functions

In [13]:
if not SKIP_PRECOMPUTE:
    import numpy as np
    import torch.nn.functional as F
    from PIL import Image
    import av


    def load_video_frames(video_path, max_frames=64):
        """Load uniformly-sampled RGB frames from a video file."""
        frames = []
        try:
            container = av.open(video_path)
            stream = container.streams.video[0]
            total = stream.frames or 0

            if total > 0 and total > max_frames:
                indices = set(np.linspace(0, total - 1, max_frames, dtype=int))
            else:
                indices = None

            for idx, frame in enumerate(container.decode(video=0)):
                if indices is not None and idx not in indices:
                    continue
                frames.append(frame.to_ndarray(format="rgb24"))
                if len(frames) >= max_frames:
                    break
            container.close()
        except Exception as e:
            print(f"Error loading {video_path}: {e}")
            return None

        return frames if len(frames) >= 8 else None


    @torch.no_grad()
    def compute_vjepa_embedding(frames):
        """V-JEPA embedding from video frames. Returns (1024,) float32 tensor."""
        inputs = vjepa_processor(frames, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = vjepa_model(**inputs)
        emb = outputs.last_hidden_state.squeeze(0).mean(dim=0)
        return emb.float().cpu()


    @torch.no_grad()
    def compute_clip_image_embedding(frames, sample_frames=8):
        """CLIP image embedding averaged over sampled frames. Returns (768,) float32 tensor."""
        n = len(frames)
        indices = np.linspace(0, n - 1, min(sample_frames, n), dtype=int)
        sampled = [frames[i] for i in indices]

        embeddings = []
        for frame in sampled:
            img = Image.fromarray(frame)
            inputs = clip_processor(images=img, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            emb = clip_model.get_image_features(**inputs)
            embeddings.append(emb.pooler_output.squeeze(0))

        avg = torch.stack(embeddings).mean(dim=0)
        return F.normalize(avg, dim=0).float().cpu()


    @torch.no_grad()
    def compute_clip_text_embeddings(caption_list):
        """CLIP text embeddings for captions. Returns (num_captions, 768) float32 tensor."""
        inputs = clip_processor(text=caption_list, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        emb = clip_model.get_text_features(**inputs)
        return F.normalize(emb.pooler_output, dim=-1).float().cpu()


    # Quick sanity check
    test_frames = load_video_frames(os.path.join(DATA_DIR, "videos", f"{video_ids[0]}.mp4"), max_frames=16)
    if test_frames:
        print(f"Loaded {len(test_frames)} test frames, shape: {test_frames[0].shape}")
        test_jepa = compute_vjepa_embedding(test_frames)
        test_clip = compute_clip_image_embedding(test_frames)
        print(f"V-JEPA embedding: {test_jepa.shape}, CLIP embedding: {test_clip.shape}")
        print("Sanity check passed!")

else:
    print("Skipped (embeddings already exist)")

Loaded 16 test frames, shape: (240, 320, 3)
V-JEPA embedding: torch.Size([1024]), CLIP embedding: torch.Size([768])
Sanity check passed!


## 6. Compute All Embeddings

This is the main loop. Progress bar shows per-video status with timing estimates.

In [14]:
if not SKIP_PRECOMPUTE:
    from tqdm.auto import tqdm
    import time

    max_captions = CONFIG["max_captions"]
    max_frames = CONFIG["max_frames"]
    clip_sample = CONFIG["clip_sample_frames"]

    jepa_list = []
    clip_image_list = []
    clip_text_list = []
    valid_ids = []
    skipped = 0

    pbar = tqdm(video_ids, desc="Computing embeddings")
    for vid_id in pbar:
        video_path = os.path.join(DATA_DIR, "videos", f"{vid_id}.mp4")
        frames = load_video_frames(video_path, max_frames=max_frames)
        if frames is None:
            skipped += 1
            continue

        vid_caps = captions.get(vid_id, [])
        if not vid_caps:
            skipped += 1
            continue

        # Pad/truncate captions
        vid_caps = vid_caps[:max_captions]
        while len(vid_caps) < max_captions:
            vid_caps.append(vid_caps[-1])

        t0 = time.time()
        jepa_emb = compute_vjepa_embedding(frames)
        clip_img_emb = compute_clip_image_embedding(frames, sample_frames=clip_sample)
        clip_txt_emb = compute_clip_text_embeddings(vid_caps)
        dt = time.time() - t0

        jepa_list.append(jepa_emb)
        clip_image_list.append(clip_img_emb)
        clip_text_list.append(clip_txt_emb)
        valid_ids.append(vid_id)

        pbar.set_postfix({"done": len(valid_ids), "skipped": skipped, "last": f"{dt:.1f}s"})

    print(f"\nProcessed {len(valid_ids)} videos, skipped {skipped}")
else:
    print("Skipped (embeddings already exist)")

Computing embeddings:   0%|          | 0/500 [00:00<?, ?it/s]


Processed 500 videos, skipped 0


## 7. Save to HDF5

Saves all embeddings with train/val/test splits compatible with our local training pipeline.

In [15]:
if not SKIP_PRECOMPUTE:
    import h5py

    # Stack into arrays
    jepa_arr = torch.stack(jepa_list).numpy()       # (N, 1024)
    clip_img_arr = torch.stack(clip_image_list).numpy()  # (N, 768)
    clip_txt_arr = torch.stack(clip_text_list).numpy()   # (N, max_captions, 768)

    N = len(valid_ids)
    print(f"Total samples: {N}")
    print(f"V-JEPA shape: {jepa_arr.shape}")
    print(f"CLIP image shape: {clip_img_arr.shape}")
    print(f"CLIP text shape: {clip_txt_arr.shape}")

    # Split: 70% train, 15% val, 15% test
    train_end = int(N * 0.70)
    val_end = int(N * 0.85)

    splits = {
        "train": (0, train_end),
        "val": (train_end, val_end),
        "test": (val_end, N),
    }

    output_path = CONFIG["output_path"]
    with h5py.File(output_path, "w") as f:
        for split_name, (start, end) in splits.items():
            grp = f.create_group(split_name)
            grp.create_dataset("vjepa", data=jepa_arr[start:end])
            grp.create_dataset("clip_image", data=clip_img_arr[start:end])
            grp.create_dataset("clip_text", data=clip_txt_arr[start:end])
            grp.create_dataset("video_ids", data=[vid.encode() for vid in valid_ids[start:end]])
            print(f"  {split_name}: {end - start} samples")

    file_size = os.path.getsize(output_path) / 1e6
    print(f"\nSaved to {output_path} ({file_size:.1f} MB)")
else:
    print("Skipped (embeddings already exist)")

Total samples: 500
V-JEPA shape: (500, 1024)
CLIP image shape: (500, 768)
CLIP text shape: (500, 5, 768)
  train: 350 samples
  val: 75 samples
  test: 75 samples

Saved to msrvtt_embeddings.h5 (11.3 MB)


## 8. Verify & Summary

Quick sanity check on saved embeddings + download instructions.

In [16]:
if not SKIP_PRECOMPUTE:
    # Verify the saved file
    with h5py.File(output_path, "r") as f:
        print("HDF5 contents:")
        for split in ["train", "val", "test"]:
            grp = f[split]
            n = grp["vjepa"].shape[0]
            print(f"  {split}: {n} samples")
            print(f"    vjepa:      {grp['vjepa'].shape} {grp['vjepa'].dtype}")
            print(f"    clip_image: {grp['clip_image'].shape} {grp['clip_image'].dtype}")
            print(f"    clip_text:  {grp['clip_text'].shape} {grp['clip_text'].dtype}")

            # Check embedding norms (should be ~1.0 for CLIP, variable for V-JEPA)
            jepa_norms = np.linalg.norm(grp["vjepa"][:5], axis=1)
            clip_norms = np.linalg.norm(grp["clip_image"][:5], axis=1)
            print(f"    vjepa norms (first 5):  {jepa_norms.round(3)}")
            print(f"    clip norms (first 5):   {clip_norms.round(3)}")

    print(f"\n--- DONE ---")
    print(f"File: {output_path} ({os.path.getsize(output_path)/1e6:.1f} MB)")
    print(f"Total: {N} video embeddings with paired CLIP image + text embeddings")
    print(f"\nTo use locally, download this file and place it at:")
    print(f"  LOGOS/PoCs/jepa_clip_translator/data/msrvtt_embeddings.h5")
    print(f"\nThen run training:")
    print(f"  python train.py --embeddings data/msrvtt_embeddings.h5 --verbose")
else:
    print("Skipped (embeddings already exist)")

HDF5 contents:
  train: 350 samples
    vjepa:      (350, 1024) float32
    clip_image: (350, 768) float32
    clip_text:  (350, 5, 768) float32
    vjepa norms (first 5):  [48.977 50.126 48.533 54.005 52.066]
    clip norms (first 5):   [1. 1. 1. 1. 1.]
  val: 75 samples
    vjepa:      (75, 1024) float32
    clip_image: (75, 768) float32
    clip_text:  (75, 5, 768) float32
    vjepa norms (first 5):  [51.123 52.139 55.488 50.344 48.09 ]
    clip norms (first 5):   [1. 1. 1. 1. 1.]
  test: 75 samples
    vjepa:      (75, 1024) float32
    clip_image: (75, 768) float32
    clip_text:  (75, 5, 768) float32
    vjepa norms (first 5):  [46.629 51.226 51.032 48.974 52.711]
    clip norms (first 5):   [1. 1. 1. 1. 1.]

--- DONE ---
File: msrvtt_embeddings.h5 (11.3 MB)
Total: 500 video embeddings with paired CLIP image + text embeddings

To use locally, download this file and place it at:
  LOGOS/PoCs/jepa_clip_translator/data/msrvtt_embeddings.h5

Then run training:
  python train.py --emb

In [17]:
if not SKIP_PRECOMPUTE:
    # Download the embeddings file (Colab only)
    try:
        from google.colab import files
        files.download(output_path)
        print("Download started!")
    except ImportError:
        print("Not running in Colab — copy the file manually:")
        print(f"  scp <host>:{os.path.abspath(output_path)} LOGOS/PoCs/jepa_clip_translator/data/")
else:
    print("Skipped (embeddings already exist)")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started!


---

# Part 2: Training the JEPA → CLIP Translator

Now that we have embeddings, we train translator models to map V-JEPA (1024-dim) → CLIP (768-dim).

The coordinator runs multiple rounds:
1. **Round 1**: Linear, MLP, and Residual baselines
2. **Round 2+**: Mutated variants of the best config so far
3. Stops when convergence is detected or max rounds reached

In [ ]:
import shutil
from google.colab import drive

drive.mount("/content/drive")

# Copy embeddings from Drive to local runtime for fast I/O
DRIVE_PATH = "/content/drive/MyDrive/msrvtt_embeddings.h5"  # adjust if in a subfolder
LOCAL_PATH = "msrvtt_embeddings.h5"

shutil.copy(DRIVE_PATH, LOCAL_PATH)
print(f"Copied {DRIVE_PATH} → {LOCAL_PATH} ({os.path.getsize(LOCAL_PATH)/1e6:.1f} MB)")

## 9. Config, Models, Losses

All model definitions and config dataclasses inlined for self-contained execution.

In [ ]:
import json
import random
import uuid
import logging
from dataclasses import dataclass, field, asdict
from typing import Literal, Callable

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# ── Config dataclasses ──────────────────────────────────────────────

@dataclass
class ArchitectureConfig:
    type: Literal["linear", "mlp", "residual"] = "linear"
    hidden_dim: int = 512
    num_blocks: int = 0
    dropout: float = 0.1
    activation: str = "gelu"

@dataclass
class TrainingConfig:
    optimizer: str = "adamw"
    lr: float = 3e-4
    weight_decay: float = 0.05
    batch_size: int = 64
    max_epochs: int = 50
    early_stop_patience: int = 7
    grad_clip: float = 1.0

@dataclass
class LossConfig:
    type: Literal["mse", "cosine", "contrastive", "combined"] = "combined"
    contrastive_weight: float = 0.7
    cosine_weight: float = 0.3
    temperature: float = 0.07

@dataclass
class DataConfig:
    subset_size: int = 500
    val_fraction: float = 0.15
    clip_sample_frames: int = 8

@dataclass
class ExperimentConfig:
    experiment_id: str = "exp_001"
    architecture: ArchitectureConfig = field(default_factory=ArchitectureConfig)
    training: TrainingConfig = field(default_factory=TrainingConfig)
    loss: LossConfig = field(default_factory=LossConfig)
    data: DataConfig = field(default_factory=DataConfig)
    vjepa_dim: int = 1024
    clip_dim: int = 768

    def to_dict(self):
        return asdict(self)

    @classmethod
    def from_dict(cls, d):
        return cls(
            experiment_id=d.get("experiment_id", "exp_001"),
            architecture=ArchitectureConfig(**d.get("architecture", {})),
            training=TrainingConfig(**d.get("training", {})),
            loss=LossConfig(**d.get("loss", {})),
            data=DataConfig(**d.get("data", {})),
            vjepa_dim=d.get("vjepa_dim", 1024),
            clip_dim=d.get("clip_dim", 768),
        )

# ── Translator models ───────────────────────────────────────────────

class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, dim * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim * 4, dim), nn.Dropout(dropout),
        )
    def forward(self, x):
        return x + self.ff(self.norm(x))

class LinearTranslator(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
    def forward(self, x):
        return F.normalize(self.linear(x), dim=-1)

class MLPTranslator(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

class ResidualTranslator(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim, num_blocks, dropout):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout),
        )
        self.blocks = nn.ModuleList([ResidualBlock(hidden_dim, dropout) for _ in range(num_blocks)])
        self.output_proj = nn.Sequential(nn.LayerNorm(hidden_dim), nn.Linear(hidden_dim, output_dim))
        self.apply(self._init_weights)
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
    def forward(self, x):
        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x)
        x = self.output_proj(x)
        return F.normalize(x, dim=-1)

def build_translator(cfg, input_dim, output_dim):
    if cfg.type == "linear":
        return LinearTranslator(input_dim, output_dim)
    elif cfg.type == "mlp":
        return MLPTranslator(input_dim, output_dim, cfg.hidden_dim, cfg.dropout)
    elif cfg.type == "residual":
        return ResidualTranslator(input_dim, output_dim, cfg.hidden_dim, cfg.num_blocks, cfg.dropout)
    else:
        raise ValueError(f"Unknown translator type: {cfg.type}")

# ── Loss functions ──────────────────────────────────────────────────

def mse_loss(pred, target):
    return {"loss": F.mse_loss(pred, target)}

def cosine_loss(pred, target):
    pred_n = F.normalize(pred, dim=-1)
    target_n = F.normalize(target, dim=-1)
    loss = 1 - (pred_n * target_n).sum(dim=-1).mean()
    return {"loss": loss}

def contrastive_loss(pred, target, temperature=0.07):
    pred_n = F.normalize(pred, dim=-1)
    target_n = F.normalize(target, dim=-1)
    logits = pred_n @ target_n.T / temperature
    labels = torch.arange(len(pred), device=pred.device)
    loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2
    with torch.no_grad():
        acc = (logits.argmax(dim=1) == labels).float().mean().item()
    return {"loss": loss, "accuracy": acc}

def combined_loss(pred, target, temperature=0.07, contrastive_weight=0.7, cosine_weight=0.3):
    c = contrastive_loss(pred, target, temperature)
    cos = cosine_loss(pred, target)
    total = contrastive_weight * c["loss"] + cosine_weight * cos["loss"]
    return {"loss": total, "contrastive_loss": c["loss"].item(), "cosine_loss": cos["loss"].item(), "accuracy": c["accuracy"]}

def build_loss_fn(cfg):
    if cfg.type == "mse":
        return mse_loss
    elif cfg.type == "cosine":
        return cosine_loss
    elif cfg.type == "contrastive":
        return lambda p, t: contrastive_loss(p, t, cfg.temperature)
    elif cfg.type == "combined":
        return lambda p, t: combined_loss(p, t, cfg.temperature, cfg.contrastive_weight, cfg.cosine_weight)
    else:
        raise ValueError(f"Unknown loss type: {cfg.type}")

print("Config, models, and losses defined ✓")

## 10. Load Embeddings for Training

In [ ]:
import h5py
import numpy as np

# Uses LOCAL_PATH from Drive mount cell above
EMBEDDINGS_PATH = LOCAL_PATH

train_data = {}
val_data = {}
test_data = {}

with h5py.File(EMBEDDINGS_PATH, "r") as f:
    for name, store in [("train", train_data), ("val", val_data), ("test", test_data)]:
        grp = f[name]
        store["jepa"] = torch.tensor(grp["vjepa"][:], dtype=torch.float32)
        store["clip_image"] = torch.tensor(grp["clip_image"][:], dtype=torch.float32)
        store["clip_text"] = torch.tensor(grp["clip_text"][:], dtype=torch.float32)

print(f"Train: {train_data['jepa'].shape[0]} samples")
print(f"Val:   {val_data['jepa'].shape[0]} samples")
print(f"Test:  {test_data['jepa'].shape[0]} samples")

# Move to GPU if available
train_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for store in [train_data, val_data, test_data]:
    for k in store:
        store[k] = store[k].to(train_device)
print(f"Tensors on {train_device}")

## 11. Training Function

In [ ]:
def run_experiment(cfg, train_data, val_data):
    """Train a translator model, return results dict."""
    train_jepa = train_data["jepa"]
    train_clip = train_data["clip_image"]
    val_jepa = val_data["jepa"]
    val_clip = val_data["clip_image"]

    train_loader = DataLoader(
        TensorDataset(train_jepa, train_clip),
        batch_size=cfg.training.batch_size, shuffle=True,
    )
    val_loader = DataLoader(
        TensorDataset(val_jepa, val_clip),
        batch_size=cfg.training.batch_size, shuffle=False,
    )

    model = build_translator(cfg.architecture, cfg.vjepa_dim, cfg.clip_dim).to(train_device)
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    loss_fn = build_loss_fn(cfg.loss)

    if cfg.training.optimizer.lower() == "adamw":
        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.training.lr, weight_decay=cfg.training.weight_decay)
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg.training.lr, weight_decay=cfg.training.weight_decay)

    history = []
    best_val_loss = float("inf")
    best_epoch = 0
    best_state = None
    patience_counter = 0

    print(f"\n{'='*60}")
    print(f"Experiment: {cfg.experiment_id}")
    print(f"  arch={cfg.architecture.type}  hidden={cfg.architecture.hidden_dim}  blocks={cfg.architecture.num_blocks}  dropout={cfg.architecture.dropout}")
    print(f"  loss={cfg.loss.type}  temp={cfg.loss.temperature}  cw={cfg.loss.contrastive_weight}")
    print(f"  lr={cfg.training.lr}  bs={cfg.training.batch_size}  params={num_params:,}")
    print(f"{'='*60}")

    for epoch in range(1, cfg.training.max_epochs + 1):
        # Train
        model.train()
        train_loss_sum = 0.0
        train_batches = 0
        for batch_jepa, batch_clip in train_loader:
            optimizer.zero_grad()
            pred = model(batch_jepa)
            result = loss_fn(pred, batch_clip)
            loss = result["loss"]
            loss.backward()
            if cfg.training.grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.training.grad_clip)
            optimizer.step()
            train_loss_sum += loss.item()
            train_batches += 1

            # Batch-level output
            batch_cos = F.normalize(pred.detach(), dim=-1).mul(F.normalize(batch_clip, dim=-1)).sum(-1).mean().item()
            acc_str = f"  acc={result['accuracy']:.3f}" if "accuracy" in result else ""
            print(f"  [train] batch {train_batches}/{len(train_loader)}  loss={loss.item():.4f}  cos={batch_cos:.4f}{acc_str}")

        avg_train_loss = train_loss_sum / max(train_batches, 1)

        # Validate
        model.eval()
        val_loss_sum = 0.0
        val_cos_sum = 0.0
        val_batches = 0
        with torch.no_grad():
            for batch_jepa, batch_clip in val_loader:
                pred = model(batch_jepa)
                result = loss_fn(pred, batch_clip)
                val_loss_sum += result["loss"].item()
                batch_cos = F.normalize(pred, dim=-1).mul(F.normalize(batch_clip, dim=-1)).sum(-1).mean().item()
                val_cos_sum += batch_cos
                val_batches += 1

                print(f"  [val]   batch {val_batches}/{len(val_loader)}  loss={result['loss'].item():.4f}  cos={batch_cos:.4f}")

        avg_val_loss = val_loss_sum / max(val_batches, 1)
        avg_val_cos = val_cos_sum / max(val_batches, 1)

        history.append({"epoch": epoch, "train_loss": avg_train_loss, "val_loss": avg_val_loss, "val_cosine_sim": avg_val_cos})
        print(f"  Epoch {epoch}/{cfg.training.max_epochs}  train_loss={avg_train_loss:.4f}  val_loss={avg_val_loss:.4f}  val_cos={avg_val_cos:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_epoch = epoch
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1

        if patience_counter >= cfg.training.early_stop_patience:
            print(f"  Early stopping at epoch {epoch} (patience {cfg.training.early_stop_patience} exhausted)")
            break

    return {
        "experiment_id": cfg.experiment_id,
        "val_loss": best_val_loss,
        "val_cosine_sim": history[best_epoch - 1]["val_cosine_sim"] if history else 0,
        "epochs_trained": len(history),
        "best_epoch": best_epoch,
        "config": cfg.to_dict(),
        "history": history,
        "best_state": best_state,
    }

print("run_experiment() defined ✓")

## 12. Evaluation & Retrieval Metrics

In [ ]:
def compute_retrieval_metrics(queries, targets, ks=(1, 5, 10)):
    """R@K and median rank. Correct target is at same index."""
    queries_n = F.normalize(queries.float(), dim=-1)
    targets_n = F.normalize(targets.float(), dim=-1)
    sim_matrix = queries_n @ targets_n.T
    n = sim_matrix.shape[0]
    sorted_indices = sim_matrix.argsort(dim=1, descending=True)
    correct_indices = torch.arange(n, device=sim_matrix.device).unsqueeze(1)
    ranks = (sorted_indices == correct_indices).nonzero(as_tuple=True)[1] + 1
    metrics = {}
    for k in ks:
        metrics[f"R@{k}"] = (ranks <= k).float().mean().item()
    metrics["median_rank"] = ranks.float().median().item()
    return metrics


@torch.no_grad()
def evaluate_best(result, test_data):
    """Evaluate the best checkpoint from an experiment on test data."""
    cfg = ExperimentConfig.from_dict(result["config"])
    model = build_translator(cfg.architecture, cfg.vjepa_dim, cfg.clip_dim).to(train_device)
    model.load_state_dict({k: v.to(train_device) for k, v in result["best_state"].items()})
    model.eval()

    translated = model(test_data["jepa"])

    # Image retrieval: translated JEPA vs CLIP image
    img_metrics = compute_retrieval_metrics(translated, test_data["clip_image"])

    # Text retrieval: translated JEPA vs CLIP text (first caption)
    clip_text_first = test_data["clip_text"][:, 0, :]
    txt_metrics = compute_retrieval_metrics(translated, clip_text_first)

    # Cosine sim stats
    translated_n = F.normalize(translated, dim=-1)
    clip_img_n = F.normalize(test_data["clip_image"], dim=-1)
    cosine_sims = (translated_n * clip_img_n).sum(dim=-1)

    return {
        "image_retrieval": img_metrics,
        "text_retrieval": txt_metrics,
        "cosine_sim_mean": cosine_sims.mean().item(),
        "cosine_sim_std": cosine_sims.std().item(),
    }

print("Evaluation functions defined ✓")

## 13. Coordinator: Config Generation & Search Loop

In [ ]:
def generate_round1_configs():
    """3 initial configs: linear, MLP, residual."""
    return [
        ExperimentConfig(experiment_id="exp_001_linear", architecture=ArchitectureConfig(type="linear")),
        ExperimentConfig(experiment_id="exp_002_mlp", architecture=ArchitectureConfig(type="mlp", hidden_dim=512)),
        ExperimentConfig(experiment_id="exp_003_residual", architecture=ArchitectureConfig(type="residual", hidden_dim=512, num_blocks=4)),
    ]

def _clamp(value, lo, hi):
    return max(lo, min(hi, value))

def generate_next_configs(all_results, num_configs=3, seed=None):
    """Mutate the best config so far to produce num_configs variants."""
    rng = random.Random(seed)
    best = min(all_results, key=lambda r: r["val_loss"])
    base = json.loads(json.dumps(best["config"]))

    arch_types = ["linear", "mlp", "residual"]
    loss_types = ["mse", "cosine", "contrastive", "combined"]
    configs = []

    for _ in range(num_configs):
        d = json.loads(json.dumps(base))

        # Architecture mutations
        if rng.random() < 0.3:
            d["architecture"]["type"] = rng.choice(arch_types)
        if rng.random() < 0.5:
            factor = rng.choice([0.5, 0.75, 1.0, 1.5, 2.0])
            d["architecture"]["hidden_dim"] = max(64, int(d["architecture"]["hidden_dim"] * factor))
        if rng.random() < 0.4:
            d["architecture"]["num_blocks"] = rng.randint(1, 8)
        if rng.random() < 0.4:
            d["architecture"]["dropout"] = round(_clamp(d["architecture"]["dropout"] + rng.gauss(0, 0.05), 0.0, 0.5), 3)

        # Loss mutations
        if rng.random() < 0.3:
            d["loss"]["type"] = rng.choice(loss_types)
        if rng.random() < 0.4:
            d["loss"]["temperature"] = round(_clamp(d["loss"]["temperature"] * rng.uniform(0.5, 2.0), 0.01, 0.5), 4)
        if rng.random() < 0.4:
            d["loss"]["contrastive_weight"] = round(_clamp(rng.uniform(0.3, 0.9), 0.0, 1.0), 2)
            d["loss"]["cosine_weight"] = round(1.0 - d["loss"]["contrastive_weight"], 2)

        # Training mutations
        if rng.random() < 0.5:
            d["training"]["lr"] = round(_clamp(d["training"]["lr"] * rng.uniform(0.3, 3.0), 1e-6, 1e-2), 7)
        if rng.random() < 0.3:
            d["training"]["batch_size"] = rng.choice([32, 64, 128, 256])

        d["experiment_id"] = f"exp_{uuid.uuid4().hex[:8]}"
        configs.append(ExperimentConfig.from_dict(d))

    return configs

print("Coordinator functions defined ✓")

## 14. Run the Search Loop

Runs Round 1 (3 baselines) then iterates with mutations until convergence or max rounds.

In [ ]:
MAX_ROUNDS = 5
CONFIGS_PER_ROUND = 3
CONVERGENCE_PATIENCE = 2  # rounds without improvement before stopping

all_results = []
best_val_loss = float("inf")
rounds_without_improvement = 0

for round_num in range(1, MAX_ROUNDS + 1):
    print(f"\n{'#'*60}")
    print(f"# ROUND {round_num}/{MAX_ROUNDS}")
    print(f"{'#'*60}")

    if round_num == 1:
        configs = generate_round1_configs()
    else:
        configs = generate_next_configs(all_results, num_configs=CONFIGS_PER_ROUND)

    round_best = float("inf")
    for cfg in configs:
        result = run_experiment(cfg, train_data, val_data)
        all_results.append(result)

        if result["val_loss"] < round_best:
            round_best = result["val_loss"]

    # Check for improvement
    if round_best < best_val_loss:
        best_val_loss = round_best
        rounds_without_improvement = 0
        print(f"\n✓ New best val_loss: {best_val_loss:.4f}")
    else:
        rounds_without_improvement += 1
        print(f"\n✗ No improvement (patience {rounds_without_improvement}/{CONVERGENCE_PATIENCE})")

    if rounds_without_improvement >= CONVERGENCE_PATIENCE:
        print(f"\nConverged after {round_num} rounds.")
        break

# Summary table
print(f"\n{'='*70}")
print(f"{'ID':<25} {'arch':<10} {'val_loss':>10} {'val_cos':>10} {'epochs':>8}")
print(f"{'-'*70}")
best_result = min(all_results, key=lambda r: r["val_loss"])
for r in all_results:
    marker = " ★" if r["experiment_id"] == best_result["experiment_id"] else ""
    arch = r["config"]["architecture"]["type"]
    print(f"{r['experiment_id']:<25} {arch:<10} {r['val_loss']:>10.4f} {r['val_cosine_sim']:>10.4f} {r['epochs_trained']:>8}{marker}")

print(f"\nBest: {best_result['experiment_id']} (val_loss={best_result['val_loss']:.4f})")

## 15. Evaluate Best Model on Test Set

In [ ]:
best_result = min(all_results, key=lambda r: r["val_loss"])
eval_results = evaluate_best(best_result, test_data)

print(f"Best model: {best_result['experiment_id']}")
print(f"  Architecture: {best_result['config']['architecture']['type']}")
print(f"  Hidden dim: {best_result['config']['architecture']['hidden_dim']}")
print(f"  Blocks: {best_result['config']['architecture']['num_blocks']}")
print(f"  Loss: {best_result['config']['loss']['type']}")
print(f"  LR: {best_result['config']['training']['lr']}")
print()

img = eval_results["image_retrieval"]
txt = eval_results["text_retrieval"]
print("Image retrieval (translated JEPA → CLIP image):")
print(f"  R@1={img['R@1']:.3f}  R@5={img['R@5']:.3f}  R@10={img['R@10']:.3f}  median_rank={img['median_rank']:.0f}")
print()
print("Text retrieval (translated JEPA → CLIP text):")
print(f"  R@1={txt['R@1']:.3f}  R@5={txt['R@5']:.3f}  R@10={txt['R@10']:.3f}  median_rank={txt['median_rank']:.0f}")
print()
print(f"Cosine similarity: mean={eval_results['cosine_sim_mean']:.4f} ± {eval_results['cosine_sim_std']:.4f}")

## 16. Save Best Model & Results

In [ ]:
# Save best model checkpoint
checkpoint_path = f"best_translator_{best_result['experiment_id']}.pt"
torch.save({
    "model_state_dict": best_result["best_state"],
    "config": best_result["config"],
    "val_loss": best_result["val_loss"],
    "val_cosine_sim": best_result["val_cosine_sim"],
    "eval_results": eval_results,
}, checkpoint_path)
print(f"Saved checkpoint: {checkpoint_path}")

# Save full experiment log
log_path = "experiment_log.json"
log_data = {
    "experiments": [{k: v for k, v in r.items() if k != "best_state"} for r in all_results],
    "best_experiment_id": best_result["experiment_id"],
    "eval_results": eval_results,
}
with open(log_path, "w") as f:
    json.dump(log_data, f, indent=2)
print(f"Saved experiment log: {log_path}")

# Download both files (Colab)
try:
    from google.colab import files
    files.download(checkpoint_path)
    files.download(log_path)
    print("Downloads started!")
except ImportError:
    print("Not in Colab — copy files manually")